# Validate Silver Table

In [ ]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "warehouse").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook inside the repository."
    )


def find_java_home() -> Path:
    # Spark's bundled Hadoop is incompatible with Java 24+, so pin a JDK <= 21.
    candidates = [
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    for candidate in candidates:
        if (candidate / "bin" / "java").exists():
            return candidate
    raise FileNotFoundError(
        "No JDK 17/21 found. Install one with: brew install openjdk@21"
    )


PROJECT_ROOT = find_project_root()
os.environ["JAVA_HOME"] = str(find_java_home())
# Workers must run the same Python as the driver, not whatever python3 is on PATH.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

CATALOG_DB = PROJECT_ROOT / "data" / "catalog.db"
WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"

# iceberg-spark-runtime must match PySpark's Spark/Scala version (4.1 / 2.13).
SPARK_PACKAGES = ",".join(
    [
        "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0",
        "org.xerial:sqlite-jdbc:3.49.1.0",
    ]
)

spark = (
    SparkSession.builder
    .appName("nyc-taxi-lakehouse")
    .master("local[*]")
    # Full-table dropDuplicates on 11M rows needs more than the default 1g heap.
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", SPARK_PACKAGES)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    # Same SQLite catalog the pipelines write to (pyiceberg SqlCatalog).
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    .config("spark.sql.catalog.local.uri", f"jdbc:sqlite:{CATALOG_DB}")
    .config("spark.sql.catalog.local.warehouse", WAREHOUSE_DIR.as_uri())
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

SILVER_TABLE = "local.silver.yellow_trips"
QUARANTINE_TABLE = "local.quarantine.yellow_trips_rejected"
TRANSFORM_LOG_TABLE = "local.metadata.transform_log"

MIN_PICKUP = "2025-12-31 23:00:00"
MAX_PICKUP = "2026-04-01 01:00:00"

print("Project root:", PROJECT_ROOT)
print("Java home:", os.environ["JAVA_HOME"])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/12 19:22:21 WARN Utils: Your hostname, admins-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.125 instead (on interface en0)
26/07/12 19:22:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ittipat/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ittipat/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
org.xerial#sqlite-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-83709579-b40b-4b71-b0fb-d91b543fa2e0;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
	found org.xerial#sqlite-jdbc;3.49.1.0 in central
:: resolution report :: 

Project root: /Users/ittipat/Desktop/nyc-taxi-lakehouse-demo
Java home: /opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


In [2]:
silver_df = spark.table(SILVER_TABLE)

silver_df.printSchema()
silver_count = silver_df.count()
print(f"Silver rows: {silver_count:,}")
silver_df.show(10, truncate=False)

26/07/12 19:22:25 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: long (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- _bronze_source_file: string (nullable = false)
 |-- _bronze_inges

26/07/12 19:22:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+-------------+-----------+------------------+------------------+-------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------------------+--------------------------+---------------------------+---------------------------+---------------------+-----------------+------------------+----------------------------+-------------------+------------------+--------------------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pickup_location_id|dropoff_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|_bronze_source_file            |_bronze_ingested_at       |_silver_source_month_offset|_silver_trip_fields_missing|_silver_zero_

In [3]:
silver_df.agg(
    F.sum(F.col("pickup_datetime").isNull().cast("long")).alias("missing_pickup"),
    F.sum(F.col("dropoff_datetime").isNull().cast("long")).alias("missing_dropoff"),
    F.sum(
        (F.col("dropoff_datetime") <= F.col("pickup_datetime")).cast("long")
    ).alias("dropoff_not_after_pickup"),
    F.sum(
        (F.col("pickup_datetime") < F.lit(MIN_PICKUP).cast("timestamp_ntz")).cast("long")
    ).alias("pickup_before_window"),
    F.sum(
        (F.col("pickup_datetime") > F.lit(MAX_PICKUP).cast("timestamp_ntz")).cast("long")
    ).alias("pickup_after_window"),
    F.sum(
        (F.col("trip_distance").isNull() | (F.col("trip_distance") < 0)).cast("long")
    ).alias("negative_or_null_distance"),
    F.sum(
        (F.col("fare_amount").isNull() | (F.col("fare_amount") < 0)).cast("long")
    ).alias("negative_or_null_fare"),
    F.sum(
        (F.col("total_amount").isNull() | (F.col("total_amount") < 0)).cast("long")
    ).alias("negative_or_null_total"),
    F.sum(
        (F.col("passenger_count").isNotNull() & (F.col("passenger_count") <= 0)).cast("long")
    ).alias("nonpositive_passenger_count"),
).show(truncate=False)

+--------------+---------------+------------------------+--------------------+-------------------+-------------------------+---------------------+----------------------+---------------------------+
|missing_pickup|missing_dropoff|dropoff_not_after_pickup|pickup_before_window|pickup_after_window|negative_or_null_distance|negative_or_null_fare|negative_or_null_total|nonpositive_passenger_count|
+--------------+---------------+------------------------+--------------------+-------------------+-------------------------+---------------------+----------------------+---------------------------+
|0             |0              |0                       |0                   |0                  |0                        |0                    |0                     |0                          |
+--------------+---------------+------------------------+--------------------+-------------------+-------------------------+---------------------+----------------------+---------------------------+



In [ ]:
WARNING_FLAGS = [
    "_silver_trip_fields_missing",
    "_silver_zero_distance",
    "_silver_zero_fare",
    "_silver_zero_total",
    "_silver_total_less_than_fare",
]

silver_df.agg(
    *[F.sum(F.col(flag).cast("long")).alias(flag.removeprefix("_silver_")) for flag in WARNING_FLAGS]
).show(truncate=False)

silver_df.groupBy("_silver_source_month_offset").count().orderBy(
    "_silver_source_month_offset"
).show(truncate=False)

+-------------------+-------------+---------+----------+--------------------+
|trip_fields_missing|zero_distance|zero_fare|zero_total|total_less_than_fare|
+-------------------+-------------+---------+----------+--------------------+
|3056935            |357809       |5634     |1424      |7361                |
+-------------------+-------------+---------+----------+--------------------+

+---------------------------+--------+
|_silver_source_month_offset|count   |
+---------------------------+--------+
|-1                         |32      |
|0                          |10813436|
|1                          |6       |
+---------------------------+--------+



In [ ]:
SOURCE_MONTH_RE = r"_(\d{4})-(\d{2})\.parquet$"
source_month_number = (
    F.regexp_extract(F.col("_bronze_source_file"), SOURCE_MONTH_RE, 1).cast("int") * 12
    + F.regexp_extract(F.col("_bronze_source_file"), SOURCE_MONTH_RE, 2).cast("int")
)
pickup_month_number = F.year("pickup_datetime") * 12 + F.month("pickup_datetime")

expected_flags = {
    "_silver_trip_fields_missing": (
        F.col("passenger_count").isNull()
        & F.col("ratecode_id").isNull()
        & F.col("congestion_surcharge").isNull()
        & F.col("store_and_fwd_flag").isNull()
        & F.col("airport_fee").isNull()
    ),
    "_silver_zero_distance": F.col("trip_distance") == 0.0,
    "_silver_zero_fare": F.col("fare_amount") == 0.0,
    "_silver_zero_total": F.col("total_amount") == 0.0,
    "_silver_total_less_than_fare": F.col("total_amount") < F.col("fare_amount"),
}

silver_df.agg(
    F.sum(
        (
            F.col("_silver_source_month_offset")
            != (pickup_month_number - source_month_number)
        ).cast("long")
    ).alias("source_month_offset_mismatch"),
    *[
        F.sum(
            (F.col(flag) != F.coalesce(expected, F.lit(False))).cast("long")
        ).alias(f"{flag.removeprefix('_silver_')}_mismatch")
        for flag, expected in expected_flags.items()
    ],
).show(truncate=False)

+----------------------------+----------------------------+----------------------+------------------+-------------------+-----------------------------+
|source_month_offset_mismatch|trip_fields_missing_mismatch|zero_distance_mismatch|zero_fare_mismatch|zero_total_mismatch|total_less_than_fare_mismatch|
+----------------------------+----------------------------+----------------------+------------------+-------------------+-----------------------------+
|0                           |0                           |0                     |0                 |0                  |0                            |
+----------------------------+----------------------------+----------------------+------------------+-------------------+-----------------------------+



In [ ]:
recomputed_duration_min = (
    F.col("dropoff_datetime").cast("timestamp").cast("long")
    - F.col("pickup_datetime").cast("timestamp").cast("long")
) / 60.0

silver_df.agg(
    F.sum(
        (
            F.col("trip_duration_min").isNull()
            | (F.col("trip_duration_min") <= 0)
        ).cast("long")
    ).alias("invalid_trip_duration"),
    F.sum(
        (F.abs(F.col("trip_duration_min") - recomputed_duration_min) > 1e-6).cast("long")
    ).alias("duration_mismatch"),
    F.sum(F.col("avg_speed_mph").isNull().cast("long")).alias("missing_speed"),
    F.sum((F.col("avg_speed_mph") < 0).cast("long")).alias("negative_speed"),
    F.sum((F.col("avg_speed_mph") == 0).cast("long")).alias("zero_speed"),
    F.sum((F.col("avg_speed_mph") > 100).cast("long")).alias("speed_above_100_mph"),
).show(truncate=False)

+---------------------+-----------------+-------------+--------------+----------+-------------------+
|invalid_trip_duration|duration_mismatch|missing_speed|negative_speed|zero_speed|speed_above_100_mph|
+---------------------+-----------------+-------------+--------------+----------+-------------------+
|0                    |0                |0            |0             |357809    |2193               |
+---------------------+-----------------+-------------+--------------+----------+-------------------+



In [7]:
silver_df.groupBy("store_and_fwd_flag").count().orderBy(
    F.asc_nulls_first("store_and_fwd_flag")
).show(truncate=False)

+------------------+-------+
|store_and_fwd_flag|count  |
+------------------+-------+
|NULL              |3056935|
|0                 |7747996|
|1                 |8543   |
+------------------+-------+



In [8]:
duplicate_key_columns = [
    column
    for column in silver_df.columns
    if column not in {"_bronze_ingested_at", "_silver_processed_at"}
]

distinct_count = (
    silver_df
    .select(*duplicate_key_columns)
    .dropDuplicates()
    .count()
)

print(f"Duplicate Silver rows: {silver_count - distinct_count:,}")

Duplicate Silver rows: 0


In [ ]:
(
    silver_df
    .withColumn("pickup_month", F.date_format("pickup_datetime", "yyyy-MM"))
    .groupBy("pickup_month", "_bronze_source_file")
    .agg(
        F.count("*").alias("rows"),
        F.sum((F.col("_silver_source_month_offset") != 0).cast("long")).alias("off_month"),
    )
    .orderBy("pickup_month", "_bronze_source_file")
    .show(100, truncate=False)
)

spark.sql(
    f"SELECT partition, record_count, file_count FROM {SILVER_TABLE}.partitions ORDER BY partition"
).show(truncate=False)

+------------+-------------------------------+-------+---------+
|pickup_month|_bronze_source_file            |rows   |off_month|
+------------+-------------------------------+-------+---------+
|2025-12     |yellow_tripdata_2026-01.parquet|5      |5        |
|2026-01     |yellow_tripdata_2026-01.parquet|3625155|0        |
|2026-01     |yellow_tripdata_2026-02.parquet|12     |12       |
|2026-02     |yellow_tripdata_2026-01.parquet|1      |1        |
|2026-02     |yellow_tripdata_2026-02.parquet|3319208|0        |
|2026-02     |yellow_tripdata_2026-03.parquet|15     |15       |
|2026-03     |yellow_tripdata_2026-02.parquet|3      |3        |
|2026-03     |yellow_tripdata_2026-03.parquet|3869073|0        |
|2026-04     |yellow_tripdata_2026-03.parquet|2      |2        |
+------------+-------------------------------+-------+---------+

+---------+------------+----------+
|partition|record_count|file_count|
+---------+------------+----------+
|{671}    |5           |1         |
|{672}    

In [ ]:
quarantine_df = spark.table(QUARANTINE_TABLE)

quarantine_count = quarantine_df.count()
print(f"Quarantined rows: {quarantine_count:,}")

HARD_RULE_FLAGS = [
    "_invalid_timestamp",
    "_invalid_distance",
    "_invalid_amount",
    "_invalid_passenger_count",
]
ALL_REJECTION_FLAGS = HARD_RULE_FLAGS + ["_invalid_source_month"]

quarantine_df.agg(
    *[F.sum(F.col(flag).cast("long")).alias(flag) for flag in ALL_REJECTION_FLAGS]
).show(truncate=False)

no_reason = quarantine_df.filter(
    ~F.col("_invalid_timestamp")
    & ~F.col("_invalid_distance")
    & ~F.col("_invalid_amount")
    & ~F.col("_invalid_passenger_count")
).count()
print(f"Quarantined rows without a hard-rule reason: {no_reason:,}")

quarantine_df.groupBy(*ALL_REJECTION_FLAGS).count().orderBy(F.desc("count")).show(truncate=False)

Quarantined rows: 263,732
+------------------+-----------------+---------------+------------------------+---------------------+
|_invalid_timestamp|_invalid_distance|_invalid_amount|_invalid_passenger_count|_invalid_source_month|
+------------------+-----------------+---------------+------------------------+---------------------+
|134803            |0                |88623          |40638                   |2                    |
+------------------+-----------------+---------------+------------------------+---------------------+

Quarantined rows without a hard-rule reason: 0
+------------------+-----------------+---------------+------------------------+---------------------+------+
|_invalid_timestamp|_invalid_distance|_invalid_amount|_invalid_passenger_count|_invalid_source_month|count |
+------------------+-----------------+---------------+------------------------+---------------------+------+
|true              |false            |false          |false                   |false     

In [11]:
transform_log_df = spark.table(TRANSFORM_LOG_TABLE).filter(
    F.col("target_table") == "silver.yellow_trips"
)

transform_log_df.orderBy("started_at").show(truncate=False)

unbalanced = transform_log_df.filter(
    (F.col("status") == "success")
    & (F.col("rows_read") != F.col("rows_written") + F.col("rows_rejected"))
).count()
print(f"Success entries where read != written + rejected: {unbalanced:,}")

totals = (
    transform_log_df
    .filter(F.col("status") == "success")
    .agg(
        F.sum("rows_read").alias("read"),
        F.sum("rows_rejected").alias("rejected"),
        F.sum("rows_written").alias("written"),
    )
    .collect()[0]
)

print(f"rows_read in log:      {totals['read']:,}")
print(f"rows_rejected in log:  {totals['rejected']:,}")
print(f"  quarantined (invalid): {quarantine_count:,}")
print(f"  deduplicated:          {totals['rejected'] - quarantine_count:,}")
print(f"rows_written in log:   {totals['written']:,}")
print(f"rows in Silver:        {silver_count:,}")
print(f"log matches Silver:    {totals['written'] == silver_count}")

+-------------------+-------------------------------+-------------------+-------+---------+-------------+------------+-------------------+--------------------------+--------------------------+-------------+
|source_table       |source_file                    |target_table       |status |rows_read|rows_rejected|rows_written|snapshot_id        |started_at                |finished_at               |error_message|
+-------------------+-------------------------------+-------------------+-------+---------+-------------+------------+-------------------+--------------------------+--------------------------+-------------+
|bronze.yellow_trips|yellow_tripdata_2026-01.parquet|silver.yellow_trips|success|3724889  |99728        |3625161     |2185403015926862061|2026-07-12 12:14:53.394964|2026-07-12 12:15:00.946275|NULL         |
|bronze.yellow_trips|yellow_tripdata_2026-02.parquet|silver.yellow_trips|success|3399866  |80643        |3319223     |6432008802280910061|2026-07-12 12:15:00.952944|2026-07